
## Work to do
*   make the code running for larger dataset.. look at the idxs again
*   MEG data needs to deICA methods.
*   Connect the model with a second stage of learning module.
*   Notice if there is any observable change in 1Hz and 24 Hz..
*   List item


*Today's result*


observed meaningful changes in the model's outcome with VNS..changes in the FC's has been found..



## Code and check..

In [ ]:
import h5py
import numpy as np
from scipy.io import loadmat
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, detrend
from scipy.interpolate import interp1d
import torch
import torch.nn as nn
import torch.optim as optim
# pip installs (keep in notebook cell if needed)
!pip install mne
!pip install torchdiffeq
from torchdiffeq import odeint_adjoint
import gc
import mne
import time
import sys
import os
# --- Debugging Flags ---
use_half_precision = False
debug_interval = 100

# ---------- (Your data-loading + preprocessing unchanged) ----------
try:
    file_new_raw = '/content/drive/MyDrive/CamCANData/CC120309/transdef_mf2pt2_rest_raw.fif'
    raw = mne.io.read_raw_fif(file_new_raw, preload=False)
    # Use MNE indexing with sample indices; data shape is (n_channels, n_times) slice [web:6][web:9]
    data, times = raw[322, 2000:4000]  # ECG-like channel; adjust as needed [web:6][web:9]
    ecg_data = -data[0]  # shape (n_times,) after squeeze

    mat = loadmat("/content/drive/MyDrive/CamCANData/CC120309/scout_id_309.mat")
    eeg_data = mat['Value']  # assume (regions x time)
    eeg_data = eeg_data[:, 2000:4000]  # adjust as needed [web:6][web:9]
    sc_data = loadmat('/content/drive/MyDrive/CamCANData/CC120309/SC_CC120309-27.mat')
    sc_matrix = sc_data["sc"]

    # Normalize SC
    max_val = np.max(sc_matrix)
    Sw_all = (sc_matrix / max_val) * 0.01 if max_val > 0 else sc_matrix

except FileNotFoundError as e:
    print(f"Error loading data files: {e}")
    print("Please ensure required files exist at specified paths.")
    sys.exit()

non_zero_indices_per_row = [np.nonzero(Sw_all[i, :])[0] for i in range(Sw_all.shape[0])]

def preprocess_signal(signal, fs=1000, lowcut=1.5, highcut=20):
    detrended = detrend(signal)
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(4, [low, high], btype='band')
    filtered = filtfilt(b, a, detrended)
    normalized = (filtered - np.mean(filtered)) / np.std(filtered)
    return normalized

ecg_processed = preprocess_signal(ecg_data, fs=1000, lowcut=1.5, highcut=20)
eeg_processed = np.array([preprocess_signal(row, fs=1000, lowcut=0.5, highcut=30) for row in eeg_data])

if np.any(np.isnan(ecg_processed)) or np.any(np.isnan(eeg_processed)):
    raise ValueError("NaN detected in input data after preprocessing. Check input files.")

# --- Utility Functions ---
def simulate_coupled_oscillators(T=10, dt=1/1000, alpha=1, omega1=5.01, omega2=5.1,
                                A_init=0.0001, theta_init=3.14, n=1.0, modulation=None):
    """Simulates a simple 2-oscillator system for heart model pre-training. Supports modulation."""
    N = int(T / dt)
    r1, r2, phi1, phi2 = 1.0, 1.0, 0.0, 0.0
    A12, A21 = A_init, A_init
    theta12, theta21 = theta_init, theta_init

    R1, R2, Phi1, Phi2 = np.zeros(N), np.zeros(N), np.zeros(N), np.zeros(N)
    for i in range(N):
        R1[i], R2[i], Phi1[i], Phi2[i] = r1, r2, phi1, phi2

        coupling12 = A12 * r2 * np.cos(theta12 + n * (phi2 - phi1))
        coupling21 = A21 * r1 * np.cos(theta21 + n * (phi1 - phi2))

        dr1 = alpha * r1 - r1**3 + coupling12+ 0.1*modulation[i,0] if modulation is not None else alpha * r1 - r1**3 + coupling12
        dr2 = alpha * r2 - r2**3 + coupling21+ 0.1*modulation[i,1] if modulation is not None else alpha * r2 - r2**3 + coupling21

        dphi1 = omega1 + A12 * r2 / r1 * np.sin(theta12 + n * (phi2 - phi1))
        dphi2 = omega2 + A21 * r1 / r2 * np.sin(theta21 + n * (phi1 - phi2))

        r1 += dr1 * dt
        r2 += dr2 * dt
        phi1 += dphi1 * dt
        phi2 += dphi2 * dt

    return np.stack((R1*np.cos(Phi1), R1*np.sin(Phi1), R2*np.cos(Phi2), R2*np.sin(Phi2)), axis=1)

def get_random_frequencies(num_regions, osc_per_region, low=1, high=30, seed=None):
    """Generates random initial frequencies for oscillators."""
    if seed is not None:
        np.random.seed(seed)
    total_oscillators = num_regions * osc_per_region
    freqs_hz = np.random.uniform(low, high, total_oscillators)
    return 2 * np.pi * freqs_hz

def expand_structural_connectivity(Sc_region, osc_per_region, intra_value=0.0001, seed=None):
    """Expands a regional structural connectivity matrix to the oscillator level."""
    if seed is not None:
        np.random.seed(seed)
    num_regions = Sc_region.shape[0]
    N = num_regions * osc_per_region
    Sc_full = np.zeros((N, N))
    for i in range(num_regions):
        for j in range(num_regions):
            start_i, end_i = i * osc_per_region, (i + 1) * osc_per_region
            start_j, end_j = j * osc_per_region, (j + 1) * osc_per_region
            if i == j:
                Sc_full[start_i:end_i, start_j:end_j] = intra_value
            else:
                rand_block = np.random.rand(osc_per_region, osc_per_region)
                rand_block *= Sc_region[i, j] / (rand_block.sum() + 1e-9)
                Sc_full[start_i:end_i, start_j:end_j] = rand_block
    np.fill_diagonal(Sc_full, 0.0)
    return Sc_full

def reset_weights(m):
    """Reinitialize weights of layers that have a reset_parameters method."""
    if hasattr(m, 'reset_parameters'):
        m.reset_parameters()

# --- Neural Network Models ---
class HeartModel(nn.Module):
    """A simple MLP to extract features from a simulated heart oscillator signal."""
    def __init__(self, input_dim=4, hidden_dim=100, feature_dim=50, output_dim=1):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.Sigmoid(),
            nn.Linear(hidden_dim, hidden_dim), nn.Sigmoid(),
            nn.Linear(hidden_dim, feature_dim),
        )
        self.output_layer = nn.Linear(feature_dim, output_dim)

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.output_layer(features)

    def get_features(self, x):
        return self.feature_extractor(x)

# --- VNS generator (torch) ---
def vns_signal_torch(t, amp=0.05, width=0.004,freq=24, device='cpu', dtype=torch.float32):
    """
    t     : numpy array of time (seconds)
    amp   : spike amplitude
    width : spike width in seconds (2 ms default)
    """
    #freq= 24
    period = 1.0 / freq
    phase = t % period

    half_width = width / 2.0

    # torch.zeros_like is pure PyTorch
    signal = torch.zeros_like(phase)

    # biphasic: +amp then -amp
    mask_pos = (phase >= 0) & (phase < half_width)
    mask_neg = (phase >= half_width) & (phase < width)

    signal[mask_pos] = amp
    signal[mask_neg] = -amp
    #signal = torch.tensor(signal, device=device, dtype=dtype)
    return signal



# LIF neuron model (vectorized)
def lif_neuron_torch(I, dt=1e-2, tau_m=20e-3, R=1.0, Vth=1.0, Vreset=0.0):
    V = torch.zeros_like(I)
    potential = torch.zeros_like(I)
    for i in range(1, I.shape[0]):
        dV = (-(V[i-1]) + R * I[i-1]) * (dt / tau_m)
        V[i] = V[i-1] + dV
        # Spike reset
        V[i] = torch.where(V[i] >= Vth, torch.tensor(Vreset, device=V.device), V[i])
        potential[i] = V[i]
    return potential


class ECGToOscillatorMLP(nn.Module):
    """MLP to map ECG hidden features to brain oscillator modulation, with VNS perturbation."""
    def __init__(self, input_dim=50, hidden1=64, hidden2=64, output_dim=16,
                 vns_width=0.04, vns_amp=0.0):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, output_dim)
        self.act = nn.Sigmoid()
        self.vns_width = vns_width
        self.vns_amp = vns_amp

    def forward(self, hidden_repr, t=None):
        h1 = self.act(self.fc1(hidden_repr))
        h2 = self.act(self.fc2(h1))

        if (t is not None) and (self.vns_amp != 0.0):
            # Ensure t is 1D for LIF computation
            if isinstance(t, torch.Tensor) and t.dim() > 1:
                t_1d = t.squeeze(-1)  # Remove last dimension
            else:
                t_1d = t.flatten() if hasattr(t, 'flatten') else torch.tensor(t).flatten()

            # Generate biphasic pulse (1D)
            I = vns_signal_torch(t_1d, amp=self.vns_amp, width=self.vns_width, freq=24.0)

            # LIF with proper dt computation
            if len(t_1d) > 1:
                dt = (t_1d[1] - t_1d[0]).item()
            else:
                dt = 1e-2

            lif_potential = lif_neuron_torch(I.flatten(), dt=dt)

            # Reshape to match batch size of h2
            if len(lif_potential) == h2.shape[0]:
                lif_signal = lif_potential.unsqueeze(1).expand_as(h2)
            else:
                lif_signal = lif_potential.mean() * torch.ones_like(h2)  # Fallback

            h2 = h2 + 0.1 * lif_signal  # Scale for stability

        out = self.fc3(h2)
        return out



class FeedbackMLP(nn.Module):
    """New MLP layers to process output from ECGToOscillatorMLP and feed back to heart model."""
    def __init__(self, input_dim=16, hidden_dim=64, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, mlp_output):
        return self.net(mlp_output)

# --- ODE Definition (Modified for Two-Stage Training) ---
class ODEFuc(nn.Module):
    """
    Defines the system of differential equations for the brain model.
    Modified to optionally exclude the MLP input for Stage 1 of training.
    """
    def __init__(self, mu, eta_theta, eta_omega, eta_alpha, D_function, N, Sc, mlp_model=None, hidden_repr=None):
        super().__init__()
        self.mu = mu
        self.eta_theta = eta_theta
        self.eta_omega = eta_omega
        self.eta_alpha = eta_alpha
        self.D_function = D_function
        self.N = N
        self.register_buffer('Sc', Sc)
        self.mlp_model = mlp_model
        self.hidden_repr = hidden_repr

    def forward(self, t, state):
        N = self.N
        r, phi = state[:N], state[N:2*N]
        theta = state[2*N:2*N + N**2].view(N, N)
        omega, alpha = state[2*N + N**2:3*N + N**2], state[3*N + N**2:4*N + N**2]

        # Clamp values for numerical stability
        omega_safe = torch.clamp(omega, 2 * np.pi * 0.5, 2 * np.pi * 20)
        r = torch.clamp(r, 1e-1, 2.0)
        alpha = torch.clamp(alpha, -1.0, 1.0)
        r_safe = torch.clamp(torch.where(r < 1e-6, torch.tensor(1e-6, device=r.device, dtype=r.dtype), r), 1e-5, 10.0)  # Higher min for stability
        r = r_safe
        phase_diff = torch.clamp(
            phi[None, :] / omega_safe[None, :] -
            phi[:, None] / omega_safe[:, None] +
            theta / (omega_safe[:, None] * omega_safe[None, :]), -1e2, 1e2)  # Added clamp to prevent extreme phases

        # Get target EEG signal value at time t
        D = torch.tensor(self.D_function(t.item()), device=state.device, dtype=state.dtype)
        P = torch.sum(alpha * r * torch.cos(phi))
        e = (D - P)  # Added clamp on error term

        # --- ECG Input Handling ---
        ecg_input = torch.zeros(N, device=state.device, dtype=state.dtype)
        if (self.mlp_model is not None) and (self.hidden_repr is not None):
            # Match the coarser time grid of the input signals
            t_idx = min(int(t.item() * 100), self.hidden_repr.shape[0] - 1)
            ecg_features = self.hidden_repr[t_idx].to(device=state.device, dtype=state.dtype);
            # Pass t to MLP for VNS injection
            ecg_input = self.mlp_model(ecg_features, t=t)

            # Conditional clamping based on whether VNS is being applied through the MLP
            if getattr(self.mlp_model, 'vns_amp', 0.0) == 0.0:
                # Only clamp if VNS is NOT active (baseline scenario)
                ecg_input = torch.clamp(ecg_input.squeeze(), 0.01, 5.0)
            else:
                # If VNS is active, do not clamp the MLP output
                ecg_input = ecg_input.squeeze()
                #ecg_input = torch.clamp(ecg_input.squeeze(), 0.01, 10.0)
        # --- System Dynamics ---
        coupling_r = torch.sum(torch.abs(self.Sc) * r[None, :] * torch.cos(phase_diff), dim=1)
        drdt = (self.mu - r**2) * r + coupling_r + e * torch.cos(phi) + ecg_input

        coupling_phi = torch.sum(torch.abs(self.Sc) * (r[None, :] / r_safe[:, None]) * torch.sin(phase_diff), dim=1)
        dphidt = omega + coupling_phi - (e / r_safe) * torch.sin(phi)

        # Adaptive dynamics (these are "frozen" by setting etas to 0 in Stage 2)
        dthetadt = self.eta_theta * torch.sin(phase_diff) * torch.abs(self.Sc)
        domegadt = -self.eta_omega * e * torch.sin(phi)
        dalphadt = self.eta_alpha * e * r * torch.cos(phi)

        # Clamp gradients to prevent explosions
        drdt = torch.clamp(drdt, -1e2, 1e2)
        dphidt = torch.clamp(dphidt, -1e2, 1e2)
        dthetadt = torch.clamp(dthetadt, -1e2, 1e2)
        domegadt = torch.clamp(domegadt, -1e2, 1e2)
        dalphadt = torch.clamp(dalphadt, -1e2, 1e2)

        return torch.cat([drdt, dphidt, dthetadt.flatten(), domegadt, dalphadt])

# --- ODE Solver Class ---
class TorchRevHopfNetwork:
    """A wrapper class to initialize and solve the ODE system."""
    def __init__(self, mu, eta_omega, eta_alpha, eta_theta, D_function, N, Sc, mlp_model, hidden_repr, device=None):
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.N = N
        self.ode_func = ODEFuc(
            mu=mu, eta_theta=eta_theta, eta_omega=eta_omega, eta_alpha=eta_alpha,
            D_function=D_function, N=N, Sc=torch.tensor(Sc, device=self.device, dtype=torch.float32),
            mlp_model=mlp_model, hidden_repr=hidden_repr.to(self.device) if hidden_repr is not None else None
        ).to(self.device)

    def solve(self, r0, phi0, theta0, omega0, alpha0, t_eval):
        dtype = torch.float16 if use_half_precision else torch.float32
        y0 = torch.tensor(np.concatenate([r0, phi0, theta0.flatten(), omega0, alpha0]),
                          device=self.device, dtype=dtype)
        #if not isinstance(t_eval, torch.Tensor):
        #    t_eval_tensor = torch.tensor(t_eval, device=self.device, dtype=dtype)
        #else:
        #    t_eval_tensor = t_eval.to(device=self.device, dtype=dtype)
        t_eval_tensor = torch.tensor(t_eval, device=self.device, dtype=dtype)
        # Debug: Verbose info before solve
        print(f"DEBUG: Starting ODE solve with y0 shape {y0.shape}, t_eval len {len(t_eval_tensor)}, method='rk4'")

        # Debug: Time the ODE solve
        start_time = time.time()
        sol = odeint_adjoint(self.ode_func, y0, t_eval_tensor, method='rk4', rtol=1e-5, atol=1e-7)
        solve_time = time.time() - start_time
        print(f"DEBUG: ODE solve took {solve_time:.4f} s. Sol shape: {sol.shape}")

        # Debug: Check for NaNs/infs
        if torch.any(torch.isnan(sol)) or torch.any(torch.isinf(sol)):
            print("DEBUG WARNING: NaN or Inf detected in ODE solution!")

        N = self.N
        r = sol[:, :N]
        phi = sol[:, N:2*N]
        theta = sol[:, 2*N:2*N + N**2].view(-1, N, N)
        omega = sol[:, 2*N + N**2:3*N + N**2]
        alpha = sol[:, 3*N + N**2:4*N + N**2]
        return r, phi, theta, omega, alpha

# --- Training Functions ---
def train_heart_model(ecg_target_signal, device):
    """Pre-trains the HeartModel to map a simple oscillator to an ECG signal."""
    print("--- Starting Heart Model Pre-training ---")
    heart_model = HeartModel().to(device)
    optimizer = optim.Adam(heart_model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    # Use a coarser time step for faster pre-training
    sim_osc_input = torch.tensor(simulate_coupled_oscillators(T=2, dt=0.01), dtype=torch.float32).to(device)  # Adjusted dt for fs=100
    ecg_target = torch.tensor(ecg_target_signal[::10], dtype=torch.float32).to(device).unsqueeze(1)  # Adjusted downsampling

    num_epochs = 25000
    start_time = time.time()  # Debug timing
    for epoch in range(num_epochs):
        predicted_ecg = heart_model(sim_osc_input)
        loss = criterion(predicted_ecg, ecg_target)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 2500 == 0 or (epoch + 1) % debug_interval == 0:
            print(f"Heart Model Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.6f}")
            if torch.isnan(loss):
                print("DEBUG WARNING: NaN loss in heart model training!")
    total_time = time.time() - start_time
    print(f"DEBUG: Heart model training took {total_time:.2f} s")
    print("--- Heart Model Pre-training Finished ---")
    return heart_model

def pre_train_brain_model(eeg_processed, Sw_all, target_idx, non_zero_indices_per_row, t, D_function, device):
    """
    Stage 1: Adapts brain oscillator parameters (omega, alpha, theta) to fit the
    target EEG signal without any ECG/MLP input over multiple epochs.
    """
    print("\n--- Starting Brain Dynamics Pre-training (Stage 1) ---")

    # Setup network structure and initial conditions
    connected_indices = np.unique(np.append(non_zero_indices_per_row[target_idx], target_idx))
    N_reduced_regions = len(connected_indices)
    osc_per_region = 5  # Changed to 5 oscillators per region
    N = N_reduced_regions * osc_per_region
    print(f"Reduced network has {N_reduced_regions} regions. Total oscillators: {N}")

    Sc_reduced_regional = Sw_all[np.ix_(connected_indices, connected_indices)]
    Sc_reduced_osc = expand_structural_connectivity(Sc_reduced_regional, osc_per_region, seed=42)

    omega_full = get_random_frequencies(68, osc_per_region, low=1, high=20, seed=42)
    alpha_full = np.random.uniform(0.1, 0.7, 68 * osc_per_region)
    omega0 = np.concatenate([omega_full[i * osc_per_region:(i + 1) * osc_per_region] for i in connected_indices])
    alpha0 = np.clip(np.concatenate([alpha_full[i * osc_per_region:(i + 1) * osc_per_region] for i in connected_indices]), 0.05, 0.5)
    r0 = 0.1 * np.ones(N)  # all radii start at 0.1 (adjusted for 5 oscillators)
    phi0 = np.zeros(N)

    theta_random = np.pi * (2 * np.random.rand(N, N) - 1)
    # Make it skew-symmetric: theta0[i, j] = -theta0[j, i]
    theta0 = theta_random - theta_random.T

    # Debug: Check initial conditions
    print(f"DEBUG: Initial r0 min/max: {np.min(r0)}, {np.max(r0)}")

    # Set etas for adaptation. The MLP is explicitly set to None.
    model = TorchRevHopfNetwork(
        mu=1.0, eta_omega=0.05, eta_alpha=0.005, eta_theta=0.05,
        D_function=D_function, N=N, Sc=Sc_reduced_osc,
        mlp_model=None, hidden_repr=None, device=device
    )

    criterion = nn.MSELoss()
    D_true = torch.tensor(D_function(t), device=device, dtype=torch.float32)

    num_epochs = 1500  # Changed to 400 epochs
    losses = []

    print("Solving ODE iteratively to adapt brain dynamics...")
    start_time = time.time()  # Debug timing
    for epoch in range(num_epochs):
        epoch_start = time.time()
        with torch.no_grad():
            # Solve the ODE for one "epoch"
            r, phi, theta, omega, alpha = model.solve(r0, phi0, theta0, omega0, alpha0, t)

            # Calculate the loss for this epoch to monitor adaptation
            P_out = torch.sum(alpha * r * torch.cos(phi), axis=1)
            loss = criterion(P_out, D_true)
            losses.append(loss.item())

            # Debug checks and print loss every 2 epochs
            if torch.isnan(loss):
                print(f"DEBUG WARNING: NaN loss at Brain Pre-training Epoch {epoch+1}")
            if (epoch + 1) % 2 == 0:
                print(f"Brain Pre-training Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.6f}")
                print(f"DEBUG: Epoch time: {time.time() - epoch_start:.4f} s. Running on {'GPU' if device=='cuda' else 'CPU'}")

            # Update initial conditions for the next epoch with the final state of this one
            r0, phi0 = r0, phi0
            theta0, omega0 = theta[-1].cpu().numpy(), omega[-1].cpu().numpy()
            alpha0 = alpha[-1].cpu().numpy()

            # Clean up memory
            del r, phi, theta, omega, alpha, P_out, loss
            if device == 'cuda':
                torch.cuda.empty_cache()

    total_time = time.time() - start_time
    print(f"DEBUG: Brain pre-training took {total_time:.2f} s")
    # Store the final adapted parameters
    final_params = {
        'r': r0, 'phi': phi0,
        'theta': theta0, 'omega': omega0,
        'alpha': alpha0
    }
    print("--- Brain Dynamics Pre-training Finished ---")
    return final_params, Sc_reduced_osc, N, losses

def train_mlp_on_frozen_brain(trained_heart_model, initial_brain_params, Sc_reduced_osc, N, D_function, t, device):
    """
    Stage 2: Freezes the pre-trained brain dynamics (etas=0) and trains the MLP
    to map ECG features to the brain model.
    """
    print(f"\n--- Starting MLP Training with Frozen Brain (Stage 2) ---")

    # Use the final parameters from Stage 1 as initial conditions
    r0, phi0, theta0, omega0, alpha0 = (
        initial_brain_params['r'], initial_brain_params['phi'],
        initial_brain_params['theta'], initial_brain_params['omega'],
        initial_brain_params['alpha']
    )

    # Setup models
    trained_heart_model.eval()
    mlp_model = ECGToOscillatorMLP(output_dim=N, vns_amp=0.0, vns_width=0.0).to(device)  # baseline: no VNS during training
    optimizer = torch.optim.Adam(mlp_model.parameters(), lr=1e-2)  # Slightly adjusted LR
    criterion = nn.MSELoss()

    num_epochs = 400
    losses = []
    fs = 1 / (t[1] - t[0])  # Infer sampling frequency from time vector
    simulated_ecg_input = torch.tensor(simulate_coupled_oscillators(T=t[-1]+1/fs, dt=1/fs), dtype=torch.float32).to(device)

    start_time = time.time()  # Debug timing
    for epoch in range(num_epochs):
        epoch_start = time.time()
        mlp_model.train()

        with torch.no_grad():
            hidden_repr = trained_heart_model.get_features(simulated_ecg_input)

        # Instantiate network with FROZEN etas (all 0) and the active MLP
        model = TorchRevHopfNetwork(
            mu=1.0, eta_omega=0.0, eta_alpha=0.0, eta_theta=0.0,  # FROZEN DYNAMICS
            D_function=D_function, N=N, Sc=Sc_reduced_osc,
            mlp_model=mlp_model, hidden_repr=hidden_repr, device=device
        )

        # Solve ODE. omega, alpha, theta will not adapt.
        r, phi, theta, omega, alpha = model.solve(r0, phi0, theta0, omega0, alpha0, t)

        # Calculate loss
        P_out = torch.sum(alpha * r * torch.cos(phi), axis=1)
        D_true = torch.tensor(D_function(t), device=device, dtype=torch.float32)
        loss = criterion(P_out, D_true)

        if torch.isnan(loss):
            print(f"Epoch {epoch+1}: NaN Loss encountered. Stopping training.")
            break

        # Backpropagate and optimize ONLY the MLP weights
        optimizer.zero_grad()
        loss.backward()
        #torch.nn.utils.clip_grad_norm_(mlp_model.parameters(), 1.0)
        optimizer.step()

        losses.append(loss.item())
        if (epoch + 1) % debug_interval == 0:
            print(f"MLP Training Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.6f}")
            print(f"DEBUG: Epoch time: {time.time() - epoch_start:.4f} s. Memory used: {torch.cuda.memory_allocated()/1e6:.2f} MB" if device == 'cuda' else "DEBUG: Running on CPU")

        del model, r, phi, theta, omega, alpha, hidden_repr
        torch.cuda.empty_cache()

    total_time = time.time() - start_time
    print(f"DEBUG: MLP training took {total_time:.2f} s")
    return mlp_model, losses

def train_feedback_loop(trained_heart_model, trained_mlp_model, ecg_target_signal, T=2, dt=0.01, device='cpu', num_epochs=100000, loop_iterations=1):
    """New function: Trains heart model again with feedback from MLP output through additional MLP layers. Uses actual hidden_repr from heart model."""
    print("--- Starting Feedback Loop Training (Retraining Heart with MLP Feedback) ---")

    trained_heart_model.apply(reset_weights)
    print("Heart model weights have been reinitialized.")
    trained_heart_model.train()  # Retrain the heart model # is this line letting off all the gradients calculated earlier ??
    feedback_mlp = FeedbackMLP(input_dim=trained_mlp_model.fc3.out_features).to(device)  # Input matches MLP output dim
    optimizer = optim.Adam(list(trained_heart_model.parameters()) + list(feedback_mlp.parameters()), lr=1e-3)
    criterion = nn.MSELoss()

    ecg_target = torch.tensor(ecg_target_signal[::10], dtype=torch.float32).to(device).unsqueeze(1)  # Match sampling

    start_time = time.time()
    losses = []
    for epoch in range(num_epochs):
        # Initial oscillator simulation
        sim_osc = simulate_coupled_oscillators(T=T, dt=dt)
        sim_osc_tensor = torch.tensor(sim_osc, dtype=torch.float32).to(device)

        # Loop: Generate hidden_repr from current sim_osc, get MLP output, process with feedback, modulate next sim_osc
        for _ in range(loop_iterations):
            hidden_repr = trained_heart_model.get_features(sim_osc_tensor)  # Actual hidden_repr from trained heart model
            mlp_output = trained_mlp_model(hidden_repr, t=None)  # Get output from trained MLP (post-Stage 2)
            feedback_output = feedback_mlp(mlp_output)  # Process with new feedback layers
            feedback_np = feedback_output.detach().cpu().numpy()  # For modulation

            # Feed back to simulate new oscillators
            sim_osc = simulate_coupled_oscillators(T=T, dt=dt, modulation=feedback_np)
            sim_osc_tensor = torch.tensor(sim_osc, dtype=torch.float32).to(device)

        # Final ECG prediction after loop
        predicted_ecg = trained_heart_model(sim_osc_tensor)
        loss = criterion(predicted_ecg, ecg_target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        if (epoch + 1) % debug_interval == 0:
            print(f"Feedback Loop Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.6f}")

    total_time = time.time() - start_time
    print(f"DEBUG: Feedback loop training took {total_time:.2f} s")
    print("--- Feedback Loop Training Finished ---")
    return trained_heart_model, feedback_mlp, losses

# --- Main Execution ---
if __name__ == '__main__':
    target_indices = [46, 47,50,51,28, 29, 16,17]  # You can set any list you want

    results_folder = "simulation_results"
    if not os.path.exists(results_folder):
        os.makedirs(results_folder)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"--- Using device: {device} ---")

    # Define time vector and target signal for training
    fs = 100  # Reduced from 250 for speed; reduce to 50 if underflow persists

    # Step 1: Pre-train the heart model
    trained_heart_model = train_heart_model(ecg_processed, device)

    for target_idx in target_indices:
        t_duration = 2
        t = np.arange(0, t_duration, 1/fs)
        target_signal = eeg_processed[target_idx, ::10]  # Adjusted downsampling to match fs=100
        D_function = interp1d(t, target_signal, kind='linear', bounds_error=False, fill_value=0.0)

        # Step 2 (Stage 1): Pre-train brain dynamics to learn omega, alpha, etc.
        final_brain_params, Sc_reduced_osc, N, brain_losses = pre_train_brain_model(
            eeg_processed, Sw_all, target_idx, non_zero_indices_per_row, t, D_function, device
        )

        # Step 3 (Stage 2): Freeze brain dynamics and train the MLP
        trained_mlp_model, mlp_losses = train_mlp_on_frozen_brain(
            trained_heart_model, final_brain_params, Sc_reduced_osc, N, D_function, t, device
        )

        # New Step: Retrain heart model with feedback loop from MLP output
        trained_heart_model, trained_feedback_mlp, feedback_losses = train_feedback_loop(
            trained_heart_model, trained_mlp_model, ecg_processed, T=t_duration, dt=1/fs, device=device, num_epochs=10000, loop_iterations=1
        )

        # --- Save baseline trained artifacts ---
        save_dict = {
            'heart': trained_heart_model.state_dict(),
            'mlp': trained_mlp_model.state_dict(),
            'feedback': trained_feedback_mlp.state_dict(),
            'final_params': final_brain_params,
            'Sc_reduced_osc': Sc_reduced_osc,
            'N': N
        }
        torch.save(save_dict, "trained_baseline.pt")
        print("✅ Saved trained baseline to 'trained_baseline.pt'")

        # --- Final Prediction and Plotting ---
        print("\n--- Generating final prediction plot ---")
        trained_heart_model.eval()
        trained_mlp_model.eval()
        trained_feedback_mlp.eval() # Set feedback MLP to eval mode

        with torch.no_grad():
            # Use the learned & frozen parameters as initial conditions
            r0, phi0, theta0, omega0, alpha0 = (
                final_brain_params['r'], final_brain_params['phi'],
                final_brain_params['theta'], final_brain_params['omega'],
                final_brain_params['alpha']
            )

            # Get ECG features for the final run (with baseline feedback loop)
            # This sim_osc_final is the one that was output from the last step of train_feedback_loop,
            # where the mlp_output for feedback was generated without VNS (t=None)
            sim_osc_final_baseline = simulate_coupled_oscillators(T=t_duration, dt=1/fs) # Re-run for baseline context
            sim_osc_tensor_baseline = torch.tensor(sim_osc_final_baseline, dtype=torch.float32).to(device)
            hidden_repr_final_baseline = trained_heart_model.get_features(sim_osc_tensor_baseline)
            # Ensure baseline path for ECG uses non-VNS MLP output for feedback
            mlp_output_baseline_for_feedback = trained_mlp_model(hidden_repr_final_baseline, t=None)
            feedback_output_baseline = trained_feedback_mlp(mlp_output_baseline_for_feedback)
            feedback_np_baseline = feedback_output_baseline.detach().cpu().numpy()
            sim_osc_final_baseline_modulated = simulate_coupled_oscillators(T=t_duration, dt=1/fs, modulation=feedback_np_baseline)
            sim_osc_tensor_baseline_modulated = torch.tensor(sim_osc_final_baseline_modulated, dtype=torch.float32).to(device)
            predicted_ecg_baseline = trained_heart_model(sim_osc_tensor_baseline_modulated).detach().cpu().numpy().flatten()


            # --- Baseline EEG prediction (no VNS) ---
            mlp_baseline_for_ode = ECGToOscillatorMLP(input_dim=hidden_repr_final_baseline.shape[1],
                                                      output_dim=N, vns_amp=0.0, vns_width=0.0).to(device)
            mlp_baseline_for_ode.load_state_dict(torch.load("trained_baseline.pt", weights_only=False)['mlp'])

            model_baseline = TorchRevHopfNetwork(
                mu=1.0, eta_omega=0.0, eta_alpha=0.0, eta_theta=0.0,  # FROZEN
                D_function=D_function, N=N, Sc=Sc_reduced_osc,
                mlp_model=mlp_baseline_for_ode, hidden_repr=hidden_repr_final_baseline, device=device
            )

            r_b, phi_b, theta_b, omega_b, alpha_b = model_baseline.solve(r0, phi0, theta0, omega0, alpha0, t)
            P_out_baseline = torch.sum(alpha_b * r_b * torch.cos(phi_b), axis=1).detach().cpu().numpy()


            # --- VNS prediction run for EEG and ECG ---
            VNS_SIM_width = 0.04 # Hz
            VNS_SIM_AMP = 0.04 # Increased for better visibility

            # For VNS, load the trained MLP and then set its VNS parameters for the forward pass.
            mlp_vns_for_ode = ECGToOscillatorMLP(input_dim=trained_heart_model.feature_extractor[-1].out_features,
                                                output_dim=N, vns_amp=VNS_SIM_AMP, vns_width=VNS_SIM_width).to(device)
            mlp_vns_for_ode.load_state_dict(torch.load("trained_baseline.pt", weights_only=False)['mlp'])

            # VNS-modulated EEG prediction
            model_vns = TorchRevHopfNetwork(
                mu=1.0, eta_omega=0.0, eta_alpha=0.0, eta_theta=0.0,  # FROZEN
                D_function=D_function, N=N, Sc=Sc_reduced_osc,
                mlp_model=mlp_vns_for_ode, hidden_repr=hidden_repr_final_baseline, device=device
            )

            r_v, phi_v, theta_v, omega_v, alpha_v = model_vns.solve(r0, phi0, theta0, omega0, alpha0, t)
            P_out_vns = torch.sum(alpha_v * r_v * torch.cos(phi_v), axis=1).detach().cpu().numpy()

            # VNS-modulated ECG prediction:
            # We need to use the `hidden_repr_final_baseline` (derived from original heart oscs)
            # pass it through the VNS-enabled mlp, then feedback_mlp, then heart osc sim, then heart model
            mlp_output_vns_for_feedback = mlp_vns_for_ode(hidden_repr_final_baseline, t=torch.tensor(t, dtype=torch.float32).to(device).unsqueeze(1))
            feedback_output_vns = trained_feedback_mlp(mlp_output_vns_for_feedback)
            feedback_np_vns = feedback_output_vns.detach().cpu().numpy()
            sim_osc_final_vns_modulated = simulate_coupled_oscillators(T=t_duration, dt=1/fs, modulation=feedback_np_vns)
            sim_osc_tensor_vns_modulated = torch.tensor(sim_osc_final_vns_modulated, dtype=torch.float32).to(device)
            predicted_ecg_vns = trained_heart_model(sim_osc_tensor_vns_modulated).detach().cpu().numpy().flatten()

            # Prepare target ECG for comparison (downsampled to match predicted ECG length)
            target_ecg = ecg_processed[::10]  # Downsample to 100 Hz (200 points for 2s)
            timesteps = np.linspace(0, t_duration, len(target_ecg))  # Time axis for plotting

            # Plot the results
            plt.figure(figsize=(14, 14))  # Increased height for extra subplot

            # Plot Brain Pre-training loss
            ax1 = plt.subplot(5, 1, 1)
            ax1.plot(brain_losses)
            ax1.set_title('Stage 1: Brain Dynamics Pre-training Loss')
            ax1.set_ylabel('MSE Loss')
            ax1.set_xlabel('Epoch')
            ax1.grid(True)

            # Plot MLP training loss
            ax2 = plt.subplot(5, 1, 2)
            ax2.plot(mlp_losses)
            ax2.set_title('Stage 2: MLP Training Loss')
            ax2.set_ylabel('MSE Loss')
            ax2.set_xlabel('Epoch')
            ax2.grid(True)

            # Plot Feedback Loop training loss (if applicable)
            if feedback_losses:
                ax3 = plt.subplot(5, 1, 3)
                ax3.plot(feedback_losses)
                ax3.set_title('Feedback Loop: Retraining Heart with MLP Feedback Loss')
                ax3.set_ylabel('MSE Loss')
                ax3.set_xlabel('Epoch')
                ax3.grid(True)

            # Plot final EEG prediction (Baseline vs VNS)
            ax4 = plt.subplot(5, 1, 4)
            ax4.plot(t, D_function(t), label='Actual EEG (target)', alpha=0.9, linewidth=2)
            ax4.plot(t, P_out_baseline, label='Predicted EEG - Baseline', linestyle='--', alpha=0.9, linewidth=2)
            ax4.plot(t, P_out_vns, label=f'Predicted EEG - With {VNS_SIM_width} Hz VNS', linestyle=':', alpha=0.9, linewidth=2)
            ax4.set_title('EEG Prediction: Baseline vs VNS')
            ax4.set_xlabel('Time (s)')
            ax4.set_ylabel('Normalized Amplitude')
            ax4.legend()
            ax4.grid(True)

            # New plot: Retrained HeartModel output vs. Target ECG (Baseline vs VNS)
            ax5 = plt.subplot(5, 1, 5)
            ax5.plot(timesteps, target_ecg, label='Target ECG', color='blue', linewidth=2)
            ax5.plot(timesteps, predicted_ecg_baseline, label='Predicted ECG (Baseline Feedback)', color='red', linestyle='--', linewidth=2)
            ax5.plot(timesteps, predicted_ecg_vns, label=f'Predicted ECG (VNS Feedback @ {VNS_SIM_width} Hz)', color='green', linestyle=':', linewidth=2)
            ax5.set_title('Retrained Heart Model Output vs. Target ECG (Baseline vs VNS)')
            ax5.set_xlabel('Time (s)')
            ax5.set_ylabel('Normalized Amplitude')
            ax5.legend()
            ax5.grid(True)

            plt.tight_layout()
            plt.show()

            np.savez(f"{results_folder}/simulation_outputs_idx{target_idx}.npz",
            brain_losses=brain_losses, mlp_losses=mlp_losses,
            feedback_losses=feedback_losses,
            predicted_ecg_baseline=predicted_ecg_baseline,
            predicted_ecg_vns=predicted_ecg_vns,
            P_out_baseline=P_out_baseline,
            P_out_vns=P_out_vns,
            target_ecg=target_ecg,
            target_signal=target_signal
            )
        print(f"✅ Saved all simulation outputs for idx={target_idx}.")

        print("✅ Done: shown baseline vs VNS model runs.")

Opening raw data file /content/drive/MyDrive/CamCANData/CC120309/transdef_mf2pt2_rest_raw.fif...
    Range : 29000 ... 595999 =     29.000 ...   595.999 secs
Ready.
--- Using device: cpu ---
--- Starting Heart Model Pre-training ---
Heart Model Epoch 100/25000, Loss: 0.999013
Heart Model Epoch 200/25000, Loss: 0.998057
Heart Model Epoch 300/25000, Loss: 0.988859
Heart Model Epoch 400/25000, Loss: 0.888569
Heart Model Epoch 500/25000, Loss: 0.832136
Heart Model Epoch 600/25000, Loss: 0.795082
Heart Model Epoch 700/25000, Loss: 0.767062
Heart Model Epoch 800/25000, Loss: 0.743703
Heart Model Epoch 900/25000, Loss: 0.688581
Heart Model Epoch 1000/25000, Loss: 0.662483
Heart Model Epoch 1100/25000, Loss: 0.641039
Heart Model Epoch 1200/25000, Loss: 0.621089
Heart Model Epoch 1300/25000, Loss: 0.596843
Heart Model Epoch 1400/25000, Loss: 0.569908
Heart Model Epoch 1500/25000, Loss: 0.542212
Heart Model Epoch 1600/25000, Loss: 0.495703
Heart Model Epoch 1700/25000, Loss: 0.458831
Heart Model